# 02 — Baseline: Linear Regression (pre-publish features only)

Predicts `video_view_count` using only metadata that would be available **before a video is published** — no channel-size features (`channel_subscriber_count`, `channel_view_count`, `channel_engagement_rate`), since those reflect channel authority/history, not the video itself, and no cross-country trending-overlap features (e.g. `global_hit_share` from the EDA notebook), since those are retroactive — you can't know how many countries a video will trend in before it's published.

This is trained on an 80/20 split **within** `train.parquet` only. `val.parquet` (the 2026-06-11 cutoff holdout) stays untouched for later, final comparison.

In [1]:
import sys
sys.path.append('../src')

import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

from features import build_features

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

MODELS_DIR = Path('../models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = Path('../data/processed/train.parquet')
df = pd.read_parquet(DATA_PATH)
df = build_features(df)
print(f"Loaded and featurized {len(df):,} rows")

Loaded and featurized 9,876,866 rows


## Feature set — pre-publish metadata only

**Included**: `video_category_id`, `tag_count`, `title_length`, `title_has_caps_word`, `publish_hour`, `publish_dayofweek`, `video_trending_country`.

**Explicitly excluded**: `channel_subscriber_count`, `channel_view_count`, `channel_engagement_rate` (channel-size confound — describes the channel's standing, not this video, and isn't fixed at publish time either), and anything derived from cross-country trending overlap (e.g. `global_hit_share`, `other_countries` from the EDA notebook — retroactive, only knowable after a video has already trended everywhere it's going to).

**Target**: `video_view_count` (raw, untransformed — this is the actual baseline test of a plain linear model against the real skewed target, not a log-smoothed version of the problem).

In [2]:
FEATURE_COLUMNS_RAW = [
    'video_category_id',
    'tag_count',
    'title_length',
    'title_has_caps_word',
    'publish_hour',
    'publish_dayofweek',
    'video_trending_country',
]
TARGET_COLUMN = 'video_view_count'

model_df = df[FEATURE_COLUMNS_RAW + [TARGET_COLUMN]].dropna().copy()
print(f"Rows after dropping any remaining nulls in the feature/target set: {len(model_df):,} "
      f"(from {len(df):,})")

X_raw = model_df[FEATURE_COLUMNS_RAW]
y = model_df[TARGET_COLUMN].astype('float64')
X_raw.head()

Rows after dropping any remaining nulls in the feature/target set: 9,876,866 (from 9,876,866)


,video_category_id,tag_count,title_length,title_has_caps_word,publish_hour,publish_dayofweek,video_trending_country
0,Music,17,38,1,0,4,United Arab Emirates
1,Gaming,55,39,0,11,4,United Arab Emirates
2,Film & Animation,0,96,0,12,2,United Arab Emirates
3,Gaming,18,27,0,2,5,United Arab Emirates
4,Music,17,99,0,9,2,United Arab Emirates


## One-hot encode `video_category_id` and `video_trending_country`

Using `drop_first=True` so each dummy set has a well-defined baseline category (avoids the dummy-variable trap / perfect multicollinearity, which also keeps the linear regression coefficients uniquely identified and interpretable relative to that baseline).

In [3]:
categorical_cols = ['video_category_id', 'video_trending_country']
numeric_cols = ['tag_count', 'title_length', 'title_has_caps_word', 'publish_hour', 'publish_dayofweek']

X_encoded = pd.get_dummies(X_raw, columns=categorical_cols, drop_first=True)
FEATURE_COLUMNS = list(X_encoded.columns)

dropped_category_baseline = sorted(X_raw['video_category_id'].astype(str).unique())[0]
dropped_country_baseline = sorted(X_raw['video_trending_country'].astype(str).unique())[0]

print(f"Feature matrix shape after one-hot encoding: {X_encoded.shape}")
print(f"Total features: {len(FEATURE_COLUMNS)} "
      f"({len(numeric_cols)} numeric + {len(FEATURE_COLUMNS) - len(numeric_cols)} one-hot dummies)")
print(f"Dropped (baseline) category: {dropped_category_baseline!r}")
print(f"Dropped (baseline) country: {dropped_country_baseline!r}")

Feature matrix shape after one-hot encoding: (9876866, 129)
Total features: 129 (5 numeric + 124 one-hot dummies)
Dropped (baseline) category: 'Autos & Vehicles'
Dropped (baseline) country: 'Algeria'


## Train/validation split — 80/20 WITHIN train.parquet

`random_state=42`. This is separate from `val.parquet` (the time-based holdout), which stays untouched for later.

In [4]:
X_train, X_val, y_train, y_val = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42
)
print(f"Train: {X_train.shape}")
print(f"Val:   {X_val.shape}")

Train: (7901492, 129)
Val:   (1975374, 129)


## Train LinearRegression, evaluate on the validation split

In [5]:
model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_val)

rmse = np.sqrt(mean_squared_error(y_val, y_pred))
r2 = r2_score(y_val, y_pred)

print('=== Baseline Linear Regression — validation results ===')
print(f'RMSE: {rmse:,.2f} views')
print(f'R^2:  {r2:.4f}')
print()
print(f'For reference — y_val stats:')
print(y_val.describe())

=== Baseline Linear Regression — validation results ===
RMSE: 18,400,631.06 views
R^2:  0.1472

For reference — y_val stats:
count    1.975374e+06
mean     6.594953e+06
std      1.992601e+07
min      1.000000e+00
25%      1.252130e+05
50%      5.595245e+05
75%      2.786605e+06
max      4.075187e+08
Name: video_view_count, dtype: float64


## Coefficients

**Caveat on "magnitude"**: features here are on very different natural scales — one-hot dummies are 0/1, while `tag_count`/`title_length`/`publish_hour`/`publish_dayofweek` range over tens of units — and none were standardized. So a raw coefficient is "predicted view-count change per one-unit change in that feature" (or per switching that dummy on), **not** a standardized effect size. Comparing a one-hot country coefficient against a per-hour or per-tag coefficient directly is apples-to-oranges; read magnitude within each feature type, not blindly across all of them.

In [6]:
coefs = pd.Series(model.coef_, index=FEATURE_COLUMNS).sort_values(ascending=False)

print('Intercept:', f'{model.intercept_:,.2f}')
print()
print('=== Top 15 POSITIVE coefficients ===')
print(coefs.head(15))
print()
print('=== Top 15 NEGATIVE coefficients ===')
print(coefs.tail(15).sort_values())
print()

top5_overall = coefs.reindex(coefs.abs().sort_values(ascending=False).index).head(5)
print('=== Top 5 overall, by |coefficient| ===')
print(top5_overall)

Intercept: 5,930,508.04

=== Top 15 POSITIVE coefficients ===
video_trending_country_Malta              2.233655e+07
video_category_id_Pets & Animals          2.022131e+07
video_trending_country_Luxembourg         1.880560e+07
video_trending_country_Slovenia           1.852645e+07
video_category_id_Howto & Style           1.790981e+07
video_trending_country_Cyprus             1.728982e+07
video_trending_country_North Macedonia    1.601613e+07
video_trending_country_Estonia            1.511719e+07
video_trending_country_Montenegro         1.422464e+07
video_trending_country_Georgia            1.307584e+07
video_trending_country_Latvia             1.303409e+07
video_trending_country_Uganda             1.302185e+07
video_trending_country_Zimbabwe           1.269620e+07
video_trending_country_Jamaica            1.263905e+07
video_trending_country_Lithuania          1.254337e+07
dtype: float64

=== Top 15 NEGATIVE coefficients ===
title_has_caps_word                   -3.351075e+06
video_ca

**Plain-language interpretation of the top 5**

1. **`video_trending_country_Malta` (+22.3M)** — trending in Malta is worth ~22M more predicted views than the baseline country, all else equal, and it's the single strongest coefficient in the entire model. Given what the EDA notebook already established (Malta's trending list is ~49% globally-viral padding vs. ~9% for the US), this is very likely the model picking up that same composition bias, not a real "Malta audiences engage more" signal — the linear model has no way to distinguish the two.
2. **`video_category_id_Pets & Animals` (+20.2M)** — this category predicts substantially higher views than the baseline category, consistent with it topping the category chart in the EDA notebook.
3. **`video_trending_country_Luxembourg` (+18.8M)** — same small-market composition pattern as Malta; same caveat applies.
4. **`video_trending_country_Slovenia` (+18.5M)** — same pattern again. **3 of the model's top 5 most influential coefficients are small-market country dummies**, meaning the single biggest thing this linear model has learned is "did this trend in a small market," which we already know is largely a data-composition artifact rather than a video-quality signal.
5. **`video_category_id_Howto & Style` (+17.9M)** — the other genuine category effect, again consistent with the EDA category chart.

**Overall fit**: R² = 0.147, RMSE ≈ 18.4M views (against a target with mean 6.6M, std 19.9M, and a max of 408M). The model explains only ~15% of the variance in raw view counts — expected for a plain linear model on a heavily right-skewed, outlier-dominated target with no channel-authority or engagement-history signal available pre-publish. Worth revisiting with a log-transformed target and/or Random Forest, both of which should handle this skew far better than raw linear regression.

One more notable result: `title_has_caps_word` is the single strongest **negative** coefficient (-3.35M) — videos with an all-caps word in the title predict *lower* views here, the opposite of the typical clickbait assumption. Worth treating as a real (if surprising) finding rather than dismissing it, the same way we did with Music's low category ranking in the EDA notebook.

## Save model and feature column list

In [7]:
MODEL_PATH = MODELS_DIR / 'baseline_linear_regression.pkl'
FEATURES_PATH = MODELS_DIR / 'baseline_feature_columns.json'

joblib.dump(model, MODEL_PATH)
with open(FEATURES_PATH, 'w') as f:
    json.dump(FEATURE_COLUMNS, f, indent=2)

print(f'Saved model to {MODEL_PATH}')
print(f'Saved {len(FEATURE_COLUMNS)} feature columns to {FEATURES_PATH}')

Saved model to ../models/baseline_linear_regression.pkl
Saved 129 feature columns to ../models/baseline_feature_columns.json


## Ablation: remove `video_trending_country` entirely

Tests whether country's apparent predictive power in the first run was mostly real signal, or mostly the composition artifact diagnosed in the EDA notebook (small markets' trending lists are padded with globally viral content, not higher local engagement). Everything else about the pipeline is held identical: same `model_df`, same numeric features, same `test_size=0.2, random_state=42` split — since the input row count/order is unchanged, this reproduces the *exact same* train/val row split as the first run, so the comparison is apples-to-apples on identical validation rows.

In [8]:
FEATURE_COLUMNS_RAW_V2 = [c for c in FEATURE_COLUMNS_RAW if c != 'video_trending_country']
print('v2 raw features:', FEATURE_COLUMNS_RAW_V2)

X_raw_v2 = model_df[FEATURE_COLUMNS_RAW_V2]
X_encoded_v2 = pd.get_dummies(X_raw_v2, columns=['video_category_id'], drop_first=True)
FEATURE_COLUMNS_V2 = list(X_encoded_v2.columns)

print(f"v2 feature matrix shape: {X_encoded_v2.shape}")
print(f"Total features: {len(FEATURE_COLUMNS_V2)} (v1 had {len(FEATURE_COLUMNS)})")

v2 raw features: ['video_category_id', 'tag_count', 'title_length', 'title_has_caps_word', 'publish_hour', 'publish_dayofweek']
v2 feature matrix shape: (9876866, 19)
Total features: 19 (v1 had 129)


In [9]:
X_train_v2, X_val_v2, y_train_v2, y_val_v2 = train_test_split(
    X_encoded_v2, y, test_size=0.2, random_state=42
)

# Confirm identical validation rows to the v1 split (same y, same random_state, same n_samples)
print('Same validation target values as v1 split:', y_val_v2.equals(y_val))

model_v2 = LinearRegression()
model_v2.fit(X_train_v2, y_train_v2)

y_pred_v2 = model_v2.predict(X_val_v2)
rmse_v2 = np.sqrt(mean_squared_error(y_val_v2, y_pred_v2))
r2_v2 = r2_score(y_val_v2, y_pred_v2)

print('=== v2 (no country) — validation results ===')
print(f'RMSE: {rmse_v2:,.2f} views')
print(f'R^2:  {r2_v2:.4f}')
print()
print('=== Comparison: v1 (with country) vs. v2 (no country) ===')
print(f'{"":12s} {"RMSE":>18s} {"R^2":>10s} {"n_features":>12s}')
print(f'{"v1":12s} {rmse:>18,.2f} {r2:>10.4f} {len(FEATURE_COLUMNS):>12d}')
print(f'{"v2":12s} {rmse_v2:>18,.2f} {r2_v2:>10.4f} {len(FEATURE_COLUMNS_V2):>12d}')
print()
print(f'RMSE change: {rmse_v2 - rmse:+,.2f} ({(rmse_v2 - rmse) / rmse:+.2%})')
print(f'R^2 change:  {r2_v2 - r2:+.4f} ({(r2_v2 - r2) / r2:+.2%} relative)')

Same validation target values as v1 split: True


=== v2 (no country) — validation results ===
RMSE: 18,932,914.63 views
R^2:  0.0972

=== Comparison: v1 (with country) vs. v2 (no country) ===
                           RMSE        R^2   n_features
v1                18,400,631.06     0.1472          129
v2                18,932,914.63     0.0972           19

RMSE change: +532,283.56 (+2.89%)
R^2 change:  -0.0500 (-33.99% relative)


In [10]:
coefs_v2 = pd.Series(model_v2.coef_, index=FEATURE_COLUMNS_V2).sort_values(ascending=False)

print('Intercept:', f'{model_v2.intercept_:,.2f}')
print()
print('=== Top 10 POSITIVE coefficients (v2, no country) ===')
print(coefs_v2.head(10))
print()
print('=== Top 10 NEGATIVE coefficients (v2, no country) ===')
print(coefs_v2.tail(10).sort_values())

Intercept: 11,037,574.25

=== Top 10 POSITIVE coefficients (v2, no country) ===
video_category_id_Pets & Animals          2.223882e+07
video_category_id_Howto & Style           2.023170e+07
video_category_id_Science & Technology    1.275060e+07
video_category_id_Comedy                  1.077607e+07
video_category_id_Entertainment           9.185505e+06
video_category_id_Travel & Events         8.613564e+06
video_category_id_People & Blogs          8.140072e+06
video_category_id_Sports                  6.906686e+06
video_category_id_Film & Animation        2.311405e+06
video_category_id_Education               1.856232e+06
dtype: float64

=== Top 10 NEGATIVE coefficients (v2, no country) ===
title_has_caps_word                       -2.931318e+06
video_category_id_Gaming                  -2.155003e+06
video_category_id_News & Politics         -1.561738e+06
video_category_id_Nonprofits & Activism   -1.528302e+06
video_category_id_Music                   -5.585771e+05
publish_hour        

**Interpretation**

Removing `video_trending_country` (129 → 19 features) confirmed the same validation rows via `y_val_v2.equals(y_val) == True`, so this is a clean, apples-to-apples comparison:

| | RMSE | R² | n_features |
|---|---:|---:|---:|
| v1 (with country) | 18,400,631 | 0.1472 | 129 |
| v2 (no country) | 18,932,915 | 0.0972 | 19 |
| change | +532,284 (+2.9%) | -0.0500 (-34.0% relative) | -110 |

**Country was carrying real predictive weight, but far less than its coefficient sizes suggested.** R² dropped by about a third relative (0.1472 → 0.0972) when country was removed — so it wasn't *pure* composition artifact with zero signal, some of what country captured is lost. But RMSE barely moved (+2.9%), and dropping 110 features (129→19) cost less than a third of the model's already-modest explanatory power. Given the EDA finding that country's top coefficients were dominated by small markets whose trending lists are ~30-49% globally-viral padding, the more defensible read is: **most of country's apparent power in v1 was the composition artifact, and only a modest residual is likely genuine regional signal** (e.g. general market-size/population effects still correlated with country even after removing the worst offenders).

**Top coefficients are now clean** — all 10 positive and 9 of 10 negative are category or title/timing features, no re-emerged country proxy. (Note: with only 19 total features, the top-10-positive and top-10-negative lists necessarily overlap by one — `video_category_id_Education` appears in both, since it's simply the 10th value in a 19-length sorted series either direction, not a data issue.) `Pets & Animals` and `Howto & Style` remain the top two positive drivers, matching the EDA category chart exactly. `title_has_caps_word` remains the strongest negative predictor by a wide margin (-2.93M), consistent with v1 — this result is robust to removing country, reinforcing that it's a real pattern worth keeping, not a fluke of the country-inflated model.

## Save v2 model and update the shared feature column list

In [11]:
MODEL_V2_PATH = MODELS_DIR / 'baseline_linear_regression_v2.pkl'

joblib.dump(model_v2, MODEL_V2_PATH)
with open(FEATURES_PATH, 'w') as f:
    json.dump(FEATURE_COLUMNS_V2, f, indent=2)

print(f'Saved v2 model to {MODEL_V2_PATH}')
print(f'Updated {FEATURES_PATH} with {len(FEATURE_COLUMNS_V2)} v2 feature columns')
print()
print('NOTE: baseline_feature_columns.json now reflects v2 (no country) and no longer '
      'matches baseline_linear_regression.pkl (v1, 129 features including country). '
      'v1 is preserved on disk for reference/comparison but its matching feature list '
      'is no longer saved separately — if v1 needs to be reloaded and used for '
      'prediction later, its feature list must be reconstructed from this notebook.')

Saved v2 model to ../models/baseline_linear_regression_v2.pkl
Updated ../models/baseline_feature_columns.json with 19 v2 feature columns

NOTE: baseline_feature_columns.json now reflects v2 (no country) and no longer matches baseline_linear_regression.pkl (v1, 129 features including country). v1 is preserved on disk for reference/comparison but its matching feature list is no longer saved separately — if v1 needs to be reloaded and used for prediction later, its feature list must be reconstructed from this notebook.
